In [1]:
from pyspark.sql import SparkSession
import time

In [2]:
spark = SparkSession.builder \
    .appName("Spark_Lab") \
    .master("spark://spark-master:7077") \
    .config("spark.ui.port", "4044") \
    .config("spark.executor.instances","2") \
    .config("spark.executor.core","1") \
    .config("spark.executor.memory","1g") \
    .config("spark.cores.max", "2") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/16 08:17:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
sc = spark.sparkContext
rdd = sc.textFile("/opt/spark/work-dir/skew_data.csv",4)

In [6]:
rdd.take(5)

['order_id,city,quantity,price',
 '1,Hanoi,3,346',
 '2,Hanoi,5,701',
 '3,Hanoi,4,331',
 '4,Hanoi,1,463']

In [14]:
header = rdd.first()
print(header)

order_id,city,quantity,price


In [11]:
rdd_no_header = rdd.filter(lambda x: x != header)

In [12]:
rdd_split = rdd_no_header.map(lambda x: x.split(','))

In [13]:
rdd_split.take(5)

[['1', 'Hanoi', '3', '346'],
 ['2', 'Hanoi', '5', '701'],
 ['3', 'Hanoi', '4', '331'],
 ['4', 'Hanoi', '1', '463'],
 ['5', 'Hanoi', '4', '127']]

In [17]:
revenue_by_city = rdd_split.map(
    lambda x: (x[1],int(x[2])*int(x[3]))
).reduceByKey(lambda a,b: a + b)

In [19]:
revenue_by_city.collect()

[('HCM', 4670550),
 ('CanTho', 6309399),
 ('Hanoi', 141800529),
 ('Danang', 4503415)]

In [20]:
city_frequency = rdd_split.map(
    lambda x: (x[1],1)
).reduceByKey(lambda x,y: x + y)

In [21]:
city_frequency.collect()

[('HCM', 2968), ('CanTho', 3940), ('Hanoi', 90151), ('Danang', 2941)]